# **Word2Vec**
직접 만든 Corpus와 `nn.Embedding`을 이용하여 Skip-gram 방식으로 단어 Vector를 학습해보자.


## **1.Corpus 준비하기**

* 비슷한 문맥에서 사용되는 단어를 반복해서 포함한 작은 Corpus를 직접 만든다.

In [1]:
sentences = [
    # Fruit
    "apple is sweet fruit",
    "banana is sweet fruit",
    "grape is sweet fruit",

    "apple is delicious fruit",
    "banana is delicious fruit",
    "grape is delicious fruit",

    "apple tastes fresh",
    "banana tastes fresh",
    "grape tastes fresh",

    "people eat apple",
    "people eat banana",
    "people eat grape",

    # Animal
    "dog is cute animal",
    "cat is cute animal",
    "rabbit is cute animal",

    "dog is friendly pet",
    "cat is friendly pet",
    "rabbit is friendly pet",

    "dog has soft fur",
    "cat has soft fur",
    "rabbit has soft fur",

    "people like dog",
    "people like cat",
    "people like rabbit",

    # Vehicle
    "car is fast vehicle",
    "bus is fast vehicle",
    "train is fast vehicle",

    "car carries people",
    "bus carries people",
    "train carries people",

    "car is transportation",
    "bus is transportation",
    "train is transportation",

    "people ride car",
    "people ride bus",
    "people ride train"
]

In [2]:
print("전체 문장 수 :", len(sentences))
print(*sentences[:5], sep="\n")

전체 문장 수 : 36
apple is sweet fruit
banana is sweet fruit
grape is sweet fruit
apple is delicious fruit
banana is delicious fruit


## **2. Tokenization**

문장을 띄어쓰기 기준으로 Tokenization한다.


In [3]:
# code here
tokens = [s.split() for s in sentences]

##**3. Vocabulary**

각 Token에 고유한 Token ID를 부여한다.


In [4]:
# code here
vocab = {}

for sentences in tokens:
    for word in sentences:
        if vocab.get(word) == None:
            vocab[word] = len(vocab)

In [5]:
print(vocab)
print(len(vocab))

{'apple': 0, 'is': 1, 'sweet': 2, 'fruit': 3, 'banana': 4, 'grape': 5, 'delicious': 6, 'tastes': 7, 'fresh': 8, 'people': 9, 'eat': 10, 'dog': 11, 'cute': 12, 'animal': 13, 'cat': 14, 'rabbit': 15, 'friendly': 16, 'pet': 17, 'has': 18, 'soft': 19, 'fur': 20, 'like': 21, 'car': 22, 'fast': 23, 'vehicle': 24, 'bus': 25, 'train': 26, 'carries': 27, 'transportation': 28, 'ride': 29}
30


In [6]:
# 출력을 위해서 역변환하는 딕셔너리도 하나 만들기
id_to_word = {idx : word for word, idx in vocab.items()}

In [7]:
print(id_to_word)

{0: 'apple', 1: 'is', 2: 'sweet', 3: 'fruit', 4: 'banana', 5: 'grape', 6: 'delicious', 7: 'tastes', 8: 'fresh', 9: 'people', 10: 'eat', 11: 'dog', 12: 'cute', 13: 'animal', 14: 'cat', 15: 'rabbit', 16: 'friendly', 17: 'pet', 18: 'has', 19: 'soft', 20: 'fur', 21: 'like', 22: 'car', 23: 'fast', 24: 'vehicle', 25: 'bus', 26: 'train', 27: 'carries', 28: 'transportation', 29: 'ride'}


## **4. 학습 데이터 만들기**

Skip-gram은 **중심 단어를 이용하여 주변 단어를 예측**한다.
* `window_size=2`이면 중심 단어의 앞뒤 2개 Token을 주변 단어로 사용한다.
* `(중심 단어 ID, 주변 단어 ID)` 형태의 학습 데이터를 만든다.


In [8]:
# code here
window_size = 2
#(0 -> 1), (0 -> 2), (1 -> 0), (1 -> 2), (1 -> 3), (2 -> 0), (2 -> 1), (2 -> 3), (2 -> 4)
pairs = []

for words in tokens:
    ids = [vocab[word] for word in words]
    # 조합을 만들기
    for i in range(len(ids)):
        # i번째 문자를 center로 잡기
        center = ids[i]
        # 얘로부터 왼쪽과 오른쪽에 어디까지 갈 수 있는지를 판단(범위)
        start = max(0, i - window_size)
        end = min(len(ids) - 1, i + window_size)
        # start -> end(포함)까지 -> i번 index를 center로 뒀을 때 우리가 학습해야하는 window_size만큼의 문자들
        for j in range(start, end + 1):
            if j != i:
                pairs.append((center, ids[j]))

print(pairs)



[(0, 1), (0, 2), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 1), (3, 2), (4, 1), (4, 2), (1, 4), (1, 2), (1, 3), (2, 4), (2, 1), (2, 3), (3, 1), (3, 2), (5, 1), (5, 2), (1, 5), (1, 2), (1, 3), (2, 5), (2, 1), (2, 3), (3, 1), (3, 2), (0, 1), (0, 6), (1, 0), (1, 6), (1, 3), (6, 0), (6, 1), (6, 3), (3, 1), (3, 6), (4, 1), (4, 6), (1, 4), (1, 6), (1, 3), (6, 4), (6, 1), (6, 3), (3, 1), (3, 6), (5, 1), (5, 6), (1, 5), (1, 6), (1, 3), (6, 5), (6, 1), (6, 3), (3, 1), (3, 6), (0, 7), (0, 8), (7, 0), (7, 8), (8, 0), (8, 7), (4, 7), (4, 8), (7, 4), (7, 8), (8, 4), (8, 7), (5, 7), (5, 8), (7, 5), (7, 8), (8, 5), (8, 7), (9, 10), (9, 0), (10, 9), (10, 0), (0, 9), (0, 10), (9, 10), (9, 4), (10, 9), (10, 4), (4, 9), (4, 10), (9, 10), (9, 5), (10, 9), (10, 5), (5, 9), (5, 10), (11, 1), (11, 12), (1, 11), (1, 12), (1, 13), (12, 11), (12, 1), (12, 13), (13, 1), (13, 12), (14, 1), (14, 12), (1, 14), (1, 12), (1, 13), (12, 14), (12, 1), (12, 13), (13, 1), (13, 12), (15, 1), (15, 12), (1, 15), (1,

In [9]:
print(len(pairs))

288


In [10]:
for center, context in pairs[:10]:
    print(f"{id_to_word[center]} -> {id_to_word[context]}")

apple -> is
apple -> sweet
is -> apple
is -> sweet
is -> fruit
sweet -> apple
sweet -> is
sweet -> fruit
fruit -> is
fruit -> sweet


## **5. Tensor로 변환하기**

중심 단어 ID를 입력 `X`, 주변 단어 ID를 정답 `y`로 사용한다.


In [11]:
# code here
import torch

X = torch.tensor([center for center, _ in pairs])
y = torch.tensor([center for _, context in pairs])

## **6. Skip-gram 모델**

`중심 단어 ID → Embedding → Linear → 주변 단어 예측`

* `nn.Embedding`의 Vector가 주변 단어를 잘 예측하는 방향으로 학습된다.
* 학습이 끝난 뒤 `embedding.weight`가 학습된 단어 Vector이다.


In [12]:
# code here
import torch.nn as nn

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        # 1. embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        # 2. linear layer -> 다음걸 예측하는게 아니라, 일단 유사한 단어를 찾는게 목표
        self.linear = nn.Linear(
            embedding_size, 
            vocab_size # 모든 vocab에 대해서 embedding vector를 반환
        )

    def forward(self, X):
        z = self.embedding(X)
        return self.linear(z)

## **7. 모델 학습**

* 정답 주변 단어의 Token ID를 맞히는 분류 문제이므로 `CrossEntropyLoss`를 사용한다.
* 데이터가 작기 때문에 전체 데이터를 한 번에 학습한다.
- Optimizer로는 Adam을 사용하고, 학습률은 0.001로 설정한다.
- epoch은 약 2000번정도로 설정하자.


In [13]:
# code here
model = SkipGram(
    vocab_size= len(vocab),
    embedding_size= 8
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr =0.001)

In [14]:
epochs = 2000

for epoch in range(epochs):
    model.train()
    y_pred = model(X)
    loss = criterion(y_pred, y)        # loss는 의미가 없다.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | loss {loss.item()}")

epoch 100 | loss 2.850253105163574
epoch 200 | loss 1.9706549644470215
epoch 300 | loss 1.0543832778930664
epoch 400 | loss 0.5018389225006104
epoch 500 | loss 0.22711706161499023
epoch 600 | loss 0.10911349952220917
epoch 700 | loss 0.060783643275499344
epoch 800 | loss 0.03857892006635666
epoch 900 | loss 0.02678326889872551
epoch 1000 | loss 0.019761525094509125
epoch 1100 | loss 0.015221389010548592
epoch 1200 | loss 0.012101658619940281
epoch 1300 | loss 0.009857232682406902
epoch 1400 | loss 0.008183484897017479
epoch 1500 | loss 0.006899149157106876
epoch 1600 | loss 0.005890318192541599
epoch 1700 | loss 0.005082451272755861
epoch 1800 | loss 0.0044249314814805984
epoch 1900 | loss 0.003882089862599969
epoch 2000 | loss 0.0034286698792129755


## **8. Embedding Vector 확인하기**

* `embedding.weight`에서 각 단어의 학습된 Vector를 확인할 수 있다.


In [15]:
# code here
# model.embedding.weight => 각 단어들의 vector 좌표를 확인할 수 있다.
word = "banana"
word_id = vocab[word]
print(model.embedding.weight[word_id])

tensor([-1.5792,  0.2448, -0.0426, -1.6024, -0.4561,  0.2794, -2.5011, -1.0005],
       grad_fn=<SelectBackward0>)


## **9. Cosine Similarity**

* 학습된 단어 Vector 사이의 유사도를 Cosine Similarity로 비교한다.
* 값이 1에 가까울수록 두 Vector의 방향이 비슷하다.

```
import torch.nn.functional as F
F.cosine_similarity(vector1, vector2)
``


In [16]:
# code here
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

a = (model.embedding.weight[vocab["cat"]].detach())
b = (model.embedding.weight[vocab["banana"]].detach())

na = np.array([a.detach()])
nb = np.array([b.detach()])

cosine_similarity(na, nb)

array([[0.05245299]], dtype=float32)

## **10. 비슷한 단어 찾기**

* 하나의 단어와 Cosine Similarity가 높은 단어를 찾아본다.


In [17]:
# code here
def most_similar(word, topk = 3):
    vectors = model.embedding.weight.detach()  # 모든 가중치들이 들어옴
    target = vectors[vocab[word]] # center 단어

    # 모든 다른 vector와의 거리를 비교해서 점수를 냈다
    scores = cosine_similarity(
        np.array([target.detach()]),
        vectors
    )

    # topk index
    indexes = torch.topk(torch.tensor(scores), topk + 1).indices      # 나랑 똑같은 단어 = 1.00(추후 나를 뺴야함)
    result = [(id_to_word[i.item()], scores[0][i.item()]) for i in indexes[0]][1:]
    return result

In [18]:
most_similar("delicious", 3)

[('eat', np.float32(0.7249721)),
 ('car', np.float32(0.7188869)),
 ('fast', np.float32(0.7067503))]